In [33]:
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.firefox.options import Options
import re
import pandas as pd

In [34]:
# initialize driver - Firefox window should open
driver = webdriver.Firefox()

In [35]:
# link = 'https://www.capterra.com/p/208764/Tableau/reviews/'
link = 'https://www.capterra.com/p/169053/Looker/reviews/'
# link = 'https://www.capterra.com/p/176586/Power-BI/reviews/'

# open the first page - might be asked to verify that you are not a robot - verify that manually
driver.get(link)

In [36]:
# object to save reviews
all_reviews = []

# iterate through the pages, capterra had 75 pages of PowerBI reviews, Tableau 94, Looker 13
for page_number in range(1,13):
    
    # specify url page
    url = link+'?page='+str(page_number)
    
    # load url
    driver.get(url)
    
    # extract html code
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    
    # select the tags that store the review text
    all_review_boxes = soup.find_all(attrs={'class':'flex flex-col gap-8'})
    
    # iteratre through the tags
    for review in all_review_boxes:
        
        # extract review text from tag
        review_text = review.find_all('p')
        
        # clean review text
        review_text_clean = re.sub('<p>|</p>|\\[|\\]', '', str(review_text))
        
        # extract reviewer information
        reviewer = review.find_all(attrs={'typo-10 text-neutral-90 w-full lg:w-fit'})[0].contents
        
        # store review date, review title, review text, reviewer role, reviewer industry
        all_reviews.append([review.find(attrs='typo-0 text-neutral-90').contents[0], 
                            review.find('h3').contents[0].strip('"'), 
                            review_text_clean, 
                            reviewer[2], 
                            reviewer[4]])

In [37]:
df = pd.DataFrame(all_reviews, columns=['Review date', 
                                        'Review title',
                                        'Review text',
                                        'Reviewer role', 
                                        'Reviewer industry'])

In [38]:
df

,Review date,Review title,Review text,Reviewer role,Reviewer industry
0,"August 29, 2024",Mapping Complex Relationships with Google Look...,Being a Data analyst requires a Good BI tool a...,Data Analyst,Higher Education
1,"July 29, 2025",Versatile and one size solution for data analy...,self-service and easy to use for the teams - d...,COO,Hospitality
2,"April 27, 2025",Okay product for data visualisation,Good product if you are looking for a free dat...,Head of Marketing,Marketing and Advertising
3,"March 25, 2025",Looker - best data visualization platform,Overall Looker is the best visualization platf...,Accounts Administrator,Automotive
4,"March 2, 2025","Great for Data Modeling, but with a Learning C...","You can create your own reporting dashbaords, ...",Senior Campaign Specialist,Marketing and Advertising
...,...,...,...,...,...
275,"May 17, 2019",Amazing,It has a unique data presentation that makes i...,CEO,Consumer Electronics
276,"July 30, 2019",Excellent choice,Looker is one very useful business intelligenc...,PHD Researcher,Research
277,"July 10, 2023",Data viz robsute,Un outil robuste pour supporter de nombreux da...,Consultante,Marketing and Advertising
278,"May 1, 2019",For Great Insights Go With Looker,Looker has been a huge help for our entire tea...,Head of Content Marketing,Marketing and Advertising


In [39]:
df.to_excel('power_bi_capterra_reviews.xlsx', index=False)